# Combine GEE Features and PM25 Data

This notebook merges two CSV files:
- **gee_features_daily.csv**: Environmental features (temperature, pressure, wind, NO2, CO, O3, AOD)
- **pm25.csv**: PM2.5 air quality measurements and sensor locations

The merge is performed on three key fields:
- `date`: Calendar date
- `latitude`: Geographic latitude coordinate
- `longitude`: Geographic longitude coordinate

Output: **combined_gee_pm25.csv** containing all features and PM2.5 levels for each location on each date.

## Step 0: Import Libraries and Setup Paths

In [2]:
import pandas as pd
import os
from pathlib import Path

# Get the base directory (workspace root)
notebook_dir = Path.cwd()
# If running from scripts folder, go up one level
if notebook_dir.name == 'notebooks':
    base_dir = notebook_dir.parent
else:
    base_dir = notebook_dir

data_dir = base_dir / 'data'

print(f"Base directory: {base_dir}")
print(f"Data directory: {data_dir}")
print(f"\nData files present:")
for file in data_dir.glob('*.csv'):
    print(f"  - {file.name}")

Base directory: c:\Users\sarah\Desktop\Groupe Project\pm25-air-quality
Data directory: c:\Users\sarah\Desktop\Groupe Project\pm25-air-quality\data

Data files present:
  - gee_features_daily.csv
  - pm25.csv


## Step 1: Load Data Files

In [3]:
print("Loading data files...\n")

# Load the CSV files
pm25_df = pd.read_csv(data_dir / 'pm25.csv')
gee_df = pd.read_csv(data_dir / 'gee_features_daily.csv')

print(f"PM25 data loaded: {pm25_df.shape[0]} rows × {pm25_df.shape[1]} columns")
print(f"GEE data loaded: {gee_df.shape[0]} rows × {gee_df.shape[1]} columns")

print(f"\nPM25 columns: {list(pm25_df.columns)}")
print(f"\nGEE columns: {list(gee_df.columns)}")

Loading data files...

PM25 data loaded: 88041 rows × 5 columns
GEE data loaded: 164208 rows × 13 columns

PM25 columns: ['date', 'latitude', 'longitude', 'pm25', 'sensor_name']

GEE columns: ['date', 'datetime_utc', 'id', 'lat', 'lon', 'temperature_celsius', 'pressure_mb', 'wind_u', 'wind_v', 'NO2', 'CO', 'O3', 'AOD']


## Step 2: Preview Data

In [ ]:
print("First few rows of PM25 data:")
print(pm25_df.head())


print("\nFirst few rows of GEE data:")
print(gee_df.head())

First few rows of PM25 data:
         date   latitude  longitude  pm25              sensor_name
0  2016-11-21  38.788056  -0.697222   4.0                ONTINYENT
1  2016-11-21  39.456111  -0.375833   3.0  VALÈNCIA-PISTA DE SILLA
2  2016-11-21  39.480300  -0.336400   2.0      VALÈNCIA-POLITÈCNIC
3  2016-11-21  39.481100  -0.408300   5.0  VALÈNCIA - MOLÍ DEL SOL
4  2016-11-21  39.481111  -0.447222   2.0          QUART DE POBLET

First few rows of GEE data:
         date         datetime_utc  id        lat       lon  \
0  2016-11-21  2016-11-21 00:00:00  90  39.235833  9.115000   
1  2016-11-21  2016-11-21 00:00:00  91  39.260556  9.136389   
2  2016-11-22  2016-11-22 00:00:00  28  36.203880 -5.383710   
3  2016-11-22  2016-11-22 00:00:00  40  41.426100  2.148000   
4  2016-11-22  2016-11-22 00:00:00  51  41.424066  2.185757   

   temperature_celsius  pressure_mb    wind_u    wind_v  NO2  CO  O3    AOD  
0            15.842538   994.344313 -2.613506  2.663921  NaN NaN NaN  0.207  
1    

In [9]:
print(pm25_df.shape)
print(gee_df.shape)

(88041, 5)
(164208, 13)


## Step 3: Standardize Column Names

In [5]:
print("Standardizing column names...")
print(f"Original GEE columns: {list(gee_df.columns)}")

# Rename lat/lon to latitude/longitude in GEE data for consistency
gee_df = gee_df.rename(columns={'lat': 'latitude', 'lon': 'longitude'})

print(f"After rename: {list(gee_df.columns)}")
print("✓ Renamed: lat → latitude, lon → longitude")

Standardizing column names...
Original GEE columns: ['date', 'datetime_utc', 'id', 'lat', 'lon', 'temperature_celsius', 'pressure_mb', 'wind_u', 'wind_v', 'NO2', 'CO', 'O3', 'AOD']
After rename: ['date', 'datetime_utc', 'id', 'latitude', 'longitude', 'temperature_celsius', 'pressure_mb', 'wind_u', 'wind_v', 'NO2', 'CO', 'O3', 'AOD']
✓ Renamed: lat → latitude, lon → longitude


## Step 4: Convert Date Columns to Datetime Format

In [6]:
print("Converting date columns to datetime format...\n")

pm25_df['date'] = pd.to_datetime(pm25_df['date'], format='%Y-%m-%d')
gee_df['date'] = pd.to_datetime(gee_df['date'], format='%Y-%m-%d')

print(f"PM25 date range: {pm25_df['date'].min().date()} to {pm25_df['date'].max().date()}")
print(f"GEE date range: {gee_df['date'].min().date()} to {gee_df['date'].max().date()}")

print(f"\nPM25: {pm25_df['date'].nunique()} unique dates")
print(f"GEE: {gee_df['date'].nunique()} unique dates")

Converting date columns to datetime format...

PM25 date range: 2016-11-21 to 2026-04-16
GEE date range: 2016-11-21 to 2026-04-06

PM25: 3131 unique dates
GEE: 2393 unique dates


## Step 5: Merge Datasets on Date, Latitude, and Longitude

In [7]:
print("Merging datasets on date, latitude, and longitude...\n")
print(f"Merge type: Inner join (only matching rows in both datasets)")
print(f"Key fields: date, latitude, longitude\n")

merged_df = pd.merge(
    gee_df,
    pm25_df,
    on=['date', 'latitude', 'longitude'],
    how='inner'
)

print(f"✓ Merge completed successfully!")
print(f"Merged data shape: {merged_df.shape[0]} rows × {merged_df.shape[1]} columns")
print(f"\nColumns in merged dataset ({merged_df.shape[1]}):")
for i, col in enumerate(merged_df.columns, 1):
    print(f"  {i:2d}. {col}")

Merging datasets on date, latitude, and longitude...

Merge type: Inner join (only matching rows in both datasets)
Key fields: date, latitude, longitude

✓ Merge completed successfully!
Merged data shape: 41718 rows × 15 columns

Columns in merged dataset (15):
   1. date
   2. datetime_utc
   3. id
   4. latitude
   5. longitude
   6. temperature_celsius
   7. pressure_mb
   8. wind_u
   9. wind_v
  10. NO2
  11. CO
  12. O3
  13. AOD
  14. pm25
  15. sensor_name


## Step 6: Sort Data for Readability

In [10]:
print("Sorting data by date and coordinates...\n")

merged_df = merged_df.sort_values(['date', 'latitude', 'longitude']).reset_index(drop=True)

print("✓ Data sorted by date, latitude, longitude")
print(f"\nFirst few rows of merged data:")
print(merged_df.head())

Sorting data by date and coordinates...

✓ Data sorted by date, latitude, longitude

First few rows of merged data:
        date         datetime_utc  id   latitude  longitude  \
0 2016-11-25  2016-11-25 00:00:00   1  38.788056  -0.697222   
1 2016-11-25  2016-11-25 00:00:00  36  39.945278  -0.056389   
2 2016-11-30  2016-11-30 00:00:00  35  40.051944  -0.189722   
3 2016-11-30  2016-11-30 00:00:00  37  40.062222   0.072778   
4 2016-12-02  2016-12-02 00:00:00   1  38.788056  -0.697222   

   temperature_celsius  pressure_mb    wind_u    wind_v  NO2  CO  O3     AOD  \
0             6.493180   946.288660  0.230240  0.735303  NaN NaN NaN  0.1110   
1             9.251236   993.626994  0.783218 -0.592618  NaN NaN NaN  0.0220   
2            10.044782   981.911432 -2.211047 -1.568971  NaN NaN NaN  0.2380   
3            12.299584  1008.794766 -3.236886 -2.804465  NaN NaN NaN  0.2480   
4             9.350630   953.114180  0.953454  0.293060  NaN NaN NaN  0.0755   

        pm25            

## Step 7: Save Combined Dataset

In [11]:
print("Saving combined dataset...\n")

output_path = data_dir / 'combined_gee_pm25.csv'
merged_df.to_csv(output_path, index=False)

print(f"✓ Output saved to: {output_path}")
print(f"\nFile size: {output_path.stat().st_size / 1024:.2f} KB")

Saving combined dataset...

✓ Output saved to: c:\Users\sarah\Desktop\Groupe Project\pm25-air-quality\data\combined_gee_pm25.csv

File size: 9592.05 KB


## Summary and Verification

In [12]:
print("="*70)
print("MERGE SUMMARY")
print("="*70)

print(f"\nInput files:")
print(f"  PM25:                {pm25_df.shape[0]:,} rows")
print(f"  GEE Features:        {gee_df.shape[0]:,} rows")

print(f"\nOutput file (combined_gee_pm25.csv):")
print(f"  Rows:                {merged_df.shape[0]:,}")
print(f"  Columns:             {merged_df.shape[1]}")

print(f"\nData coverage:")
print(f"  Date range:          {merged_df['date'].min().date()} to {merged_df['date'].max().date()}")
print(f"  Unique dates:        {merged_df['date'].nunique()}")
print(f"  Unique locations:    {len(merged_df[['latitude', 'longitude']].drop_duplicates())}")

print(f"\nMissing values per column:")
missing = merged_df.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) == 0:
    print("  ✓ No missing values!")
else:
    for col, count in missing_cols.items():
        pct = (count / len(merged_df)) * 100
        print(f"  {col}: {count:,} ({pct:.1f}%)")

print("\n" + "="*70)
print("✓ Merge completed successfully!")
print("="*70)

MERGE SUMMARY

Input files:
  PM25:                88,041 rows
  GEE Features:        164,208 rows

Output file (combined_gee_pm25.csv):
  Rows:                41,718
  Columns:             15

Data coverage:
  Date range:          2016-11-25 to 2026-04-06
  Unique dates:        2305
  Unique locations:    98

Missing values per column:
  NO2: 2,410 (5.8%)
  CO: 2,410 (5.8%)
  O3: 2,451 (5.9%)

✓ Merge completed successfully!


## Sample Data Inspection

In [13]:
print("Sample rows from combined dataset:\n")
print(merged_df.sample(min(5, len(merged_df))).to_string())

print("\n\nData info:")
print(merged_df.info())

Sample rows from combined dataset:

            date         datetime_utc   id   latitude  longitude  temperature_celsius  pressure_mb    wind_u    wind_v       NO2        CO        O3       AOD      pm25  sensor_name
34078 2025-04-24  2025-04-24 00:00:00  103  39.667222  -0.234722            16.811465   998.555938 -0.067355  0.171090  0.000115  0.031282  0.151392  0.091000  9.777778  SAGUNT PORT
15222 2022-06-06  2022-06-06 00:00:00   68  43.348053   5.360652            24.172265   992.936273  3.567144 -0.930635  0.000096  0.031963  0.141989  0.066500  6.782609      FR03014
4503  2019-08-23  2019-08-23 00:00:00   69  43.530297   5.441383            23.825694   985.427922 -0.029695  1.076595  0.000124  0.032494  0.135230  0.162000  9.595833      FR03029
31415 2024-12-09  2024-12-09 00:00:00   98  41.193604   1.236701             8.373365  1005.258128  3.806142 -3.422215  0.000063  0.033664  0.147271  0.079500  2.777778     Perafort
23295 2024-01-22  2024-01-22 00:00:00   25  36.159370 